# 02. Feature Engineering & Vectorization

**Project**: CineMatch — TMDB Content-Based Movie Recommendation System  
**Objective**: Faithfully implement and validate CampusX content-based feature engineering: JSON parsing, top-3 cast extraction, director isolation, entity space collapse, tag synthesis, Porter stemming, and CountVectorizer representation.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from src.data_loader import load_raw_tmdb_data
from src.preprocessing import preprocess_tmdb_metadata
from src.feature_engineering import build_movie_tags

print('CineMatch pipeline modules imported successfully.')

CineMatch pipeline modules imported successfully.


## 1. Load Raw Merged TMDB Dataset

In [2]:
movies_path = '../data/raw/tmdb_5000_movies.csv' if os.path.exists('../data/raw/tmdb_5000_movies.csv') else 'data/raw/tmdb_5000_movies.csv'
credits_path = '../data/raw/tmdb_5000_credits.csv' if os.path.exists('../data/raw/tmdb_5000_credits.csv') else 'data/raw/tmdb_5000_credits.csv'

df_raw = load_raw_tmdb_data(movies_path=movies_path, credits_path=credits_path, merge_on='id')
print(f'Raw merged records: {df_raw.shape}')
df_raw[['id', 'title', 'genres', 'overview', 'cast', 'crew']].head(2)

Raw merged records: (4803, 23)


,id,title,genres,overview,cast,crew
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","In the 22nd century, a paraplegic Marine is di...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","Captain Barbossa, long believed to be dead, ha...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


## 2. Preprocessing & Entity Extraction

In [3]:
df_clean = preprocess_tmdb_metadata(df_raw, drop_null_overview=True, cast_top_k=3)
print(f'Cleaned records after null overview filtering: {df_clean.shape}')
sample = df_clean.iloc[0]
print('Movie:', sample['title'])
print('Genres clean:', sample['genres_clean'])
print('Genres collapsed:', sample['genres_collapsed'])
print('Top 3 Cast clean:', sample['cast_clean'])
print('Top 3 Cast collapsed:', sample['cast_collapsed'])
print('Director clean:', sample['crew_clean'])
print('Director collapsed:', sample['crew_collapsed'])

Cleaned records after null overview filtering: (4800, 32)
Movie: Avatar
Genres clean: ['Action', 'Adventure', 'Fantasy', 'Science Fiction']
Genres collapsed: ['Action', 'Adventure', 'Fantasy', 'ScienceFiction']
Top 3 Cast clean: ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']
Top 3 Cast collapsed: ['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver']
Director clean: ['James Cameron']
Director collapsed: ['JamesCameron']


## 3. Tag Combination & Porter Stemming

In [4]:
df_featured = build_movie_tags(df_clean, apply_stemming=True)
print('Sample synthesized tags for "Avatar":')
print(df_featured.iloc[0]['tags'][:300] + '...')

Sample synthesized tags for "Avatar":
in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplan...


## 4. Bag-of-Words Vectorization (CountVectorizer)

In [5]:
cv = CountVectorizer(max_features=5000, stop_words='english')
feature_matrix = cv.fit_transform(df_featured['tags'])

print(f'Feature matrix shape: {feature_matrix.shape}')
print(f'Sparsity: {100.0 * (1.0 - feature_matrix.nnz / (feature_matrix.shape[0] * feature_matrix.shape[1])):.2f}%')
vocab = cv.get_feature_names_out()
print(f'Vocabulary size: {len(vocab)}')
print('Sample vocabulary words (indices 1000-1015):', vocab[1000:1015])

Feature matrix shape: (4800, 5000)
Sparsity: 99.39%
Vocabulary size: 5000
Sample vocabulary words (indices 1000-1015): ['cop' 'cope' 'cops' 'core' 'corner' 'corp' 'corpor' 'corporation' 'corps'
 'corrupt' 'cost' 'costum' 'couldn' 'counselor' 'count']


## 5. Verification of Entity Preservation in Vocabulary

In [6]:
# Verify collapsed entities appear in vocabulary
sample_entities = ['jamescameron', 'samworthington', 'sciencefiction', 'action', 'alien']
for ent in sample_entities:
    in_vocab = ent in cv.vocabulary_
    idx = cv.vocabulary_.get(ent, -1)
    print(f'Entity "{ent}": present={in_vocab}, index={idx}')

Entity "jamescameron": present=True, index=2323
Entity "samworthington": present=False, index=-1
Entity "sciencefiction": present=False, index=-1
Entity "action": present=True, index=78
Entity "alien": present=True, index=156


## 6. Conclusions & Pipeline Export

In [7]:
print('Feature engineering pipeline successfully verified.')
print('All records have nonempty feature tags.')
print(f'Zero-length tags count: {(df_featured["tags"].str.len() == 0).sum()}')

Feature engineering pipeline successfully verified.
All records have nonempty feature tags.
Zero-length tags count: 0
